# Custom analyzers: address nodes by what they do

A config key is either a node name or a fact. Facts are written by analyzers, functions over the whole graph that return the nodes carrying their fact. The three built-in ones are `statistic_operand`, `weights_operand` and `input_conv`; this notebook writes two more.

1. A depth schedule: a different gamma for each group of convolutions, the way the BiLRP paper does it for VGG.
2. A side fact: which operand of a squeeze-and-excitation gate is the gate, so one `('detach', {'by': ...})` entry treats it as a constant.

Facts win over node names, so a fact entry does not disturb the rest of the graph.

In [1]:
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autoLRP (or `pip install -e .`)
import torch
import torch.nn as nn
import torch.nn.functional as F

import autolrp
from autolrp import LRPConfig, BASE, register_analyzer, ANALYZERS, explain, explain_summary
torch.manual_seed(0)

## 1. A depth schedule

`explain` lists nodes in walk order, output to input, so the first convolution in the plan is the last one of the network. An analyzer sees the same order.

In [2]:
cnn = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(),
    nn.Conv2d(8, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
    nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10),
).eval()
img = torch.randn(1, 3, 32, 32)

def convs(nodes):
    return [n for n in nodes if 'ConvolutionBackward' in n.name()]

@register_analyzer('deep_conv')
def deep_conv(nodes):                     # the two convolutions nearest the output
    return {n: True for n in convs(nodes)[:2]}

@register_analyzer('shallow_conv')
def shallow_conv(nodes):                  # the two nearest the input
    return {n: True for n in convs(nodes)[2:]}

schedule = LRPConfig(rule={**BASE,
                           'deep_conv':    ('gamma', {'gamma': 0.1}),
                           'shallow_conv': ('gamma', {'gamma': 0.5})})

rows = explain(cnn(autolrp.tensor(img))[0, 3], schedule)
print(explain_summary([r for r in rows if 'Conv' in r[0] or 'Addmm' in r[0]]))

count  node                  key                   what
    2  ConvolutionBackward0  deep_conv             gamma
    2  ConvolutionBackward0  shallow_conv          gamma
    1  AddmmBackward0        AddmmBackward         epsilon


In [3]:
x = autolrp.tensor(img.clone()); cnn(x)[0, 3].lrp(config=schedule); R_schedule = x.relevance.clone()
x = autolrp.tensor(img.clone()); cnn(x)[0, 3].lrp(config=LRPConfig.composite()); R_composite = x.relevance.clone()
print('schedule differs from composite:', not torch.allclose(R_schedule, R_composite))
print('sum R schedule  =', round(float(R_schedule.sum()), 4), ' (biases keep their share)')

schedule differs from composite: True
sum R schedule  = 0.0988  (biases keep their share)


An analyzer returns `{node: value}`; a bare `True` is a plain tag. The value can carry data: the built-in `statistic_operand` returns the slot (0 or 1) of the operand it names, which is what `('detach', {'by': 'statistic_operand'})` reads.

## 2. A side fact for a gate

A squeeze-and-excitation block multiplies its input by `sigmoid(fc(pool(x)))`. Under `BASE` that product splits its relevance in proportion to the two operands' magnitudes, and the gate's share flows back through the small MLP into the pooled input. If you would rather treat the gate as a constant, name it with a fact and detach it.

In [4]:
class SE(nn.Module):
    def __init__(self, c=8):
        super().__init__()
        self.fc1, self.fc2 = nn.Linear(c, c // 2), nn.Linear(c // 2, c)
    def forward(self, x):
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(x.mean((2, 3))))))
        return x * s[:, :, None, None]         # x is the left operand, the gate the right

se_net = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), SE(8),
                       nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(8, 4)).eval()

from autolrp.backward.analysis import parents

SHAPE = ('Alias', 'Unsqueeze', 'View', 'Expand', 'Reshape', 'Squeeze')

def producer(fn):
    """The first non-shape op above `fn`."""
    while fn is not None and any(s in fn.name() for s in SHAPE):
        fn = parents(fn, skip_aliases=False)[0]
    return fn

@register_analyzer('se_gate')
def se_gate(nodes):
    """A MulBackward whose right operand comes through a sigmoid: slot 1 is the gate."""
    out = {}
    for n in nodes:
        if 'MulBackward' not in n.name():
            continue
        ps = parents(n)
        if len(ps) == 2 and ps[1] is not None and 'Sigmoid' in producer(ps[1]).name():
            out[n] = 1
    return out

gated = LRPConfig(rule={**BASE, 'se_gate': ('detach', {'by': 'se_gate'})})
rows = explain(se_net(autolrp.tensor(img))[0, 1], gated)
print(explain_summary([r for r in rows if 'Mul' in r[0] or 'Sigmoid' in r[0]]))

count  node              key                   what
    1  MulBackward0      se_gate               detach_rhs
    1  SigmoidBackward0  None                  activation=passthrough


In [5]:
x = autolrp.tensor(img.clone()); se_net(x)[0, 1].lrp(config=LRPConfig()); R_split = x.relevance.clone()
x = autolrp.tensor(img.clone()); se_net(x)[0, 1].lrp(config=gated); R_gated = x.relevance.clone()
print('gate detached differs from split:', not torch.allclose(R_split, R_gated))

gate detached differs from split: True


`detach_lhs` and `detach_rhs` are positional: they zero the operand written on the left or on the right of that multiplication. The fact is what makes the entry follow the gate whichever side it is written on; `detach_rhs` in the config would have worked for this model and silently zeroed the wrong side in a model that writes `s * x`.

In [6]:
for name in ('deep_conv', 'shallow_conv', 'se_gate'):   # analyzers are global; remove them when done
    ANALYZERS.pop(name)